# 01 — Getting Connected

This notebook logs you in to OpenShift, grabs your API token, calls the AI models endpoint, and connects to S3-compatible storage (MinIO).

Run each cell from top to bottom by pressing **Shift + Enter**.

In [ ]:
import os
os.environ["UV_EXTRA_INDEX_URL"] = "https://pypi.org/simple"

# enable colour output for all print() calls in this notebook
!uv pip install rich -q
from rich import print, print_json

## Step 1 — Log in to OpenShift

Fill in your username below. The server address is detected automatically from the environment. Your password will be entered securely (it won't be saved in the notebook).

In [ ]:
import subprocess
import getpass
import os

host = os.environ["KUBERNETES_SERVICE_HOST"]
port = os.environ["KUBERNETES_SERVICE_PORT"]
OCP_SERVER = f"https://{host}:{port}"

OCP_USER = "user1"   # update to your username
OCP_PASS = getpass.getpass("Enter your OpenShift password: ")

result = subprocess.run(
    ["oc", "login", OCP_SERVER,
     "--username", OCP_USER,
     "--password", OCP_PASS,
     "--insecure-skip-tls-verify"],
    capture_output=True, text=True,
    cwd="/opt/app-root/src"
)

print(result.stdout)
if result.returncode != 0:
    print("Login failed:", result.stderr)

## Step 2 — Confirm you are logged in

In [ ]:
!oc whoami
!oc project

## Step 3 — Get your MaaS API token

The MaaS API uses its own token stored in a Kubernetes secret called `maas-secret`.

In [ ]:
import base64

result = subprocess.run(
    ["oc", "get", "secret", "maas-secret", "-o", "jsonpath={.data.token}"],
    capture_output=True, text=True
)
TOKEN = base64.b64decode(result.stdout.strip()).decode()
print("Token obtained:", TOKEN[:20], "...")

## Step 4 — List available AI models

In [ ]:
import requests

MAAS_API_URL = "https://maas.apps.ocp.cloud.rhai-tmm.dev"

response = requests.get(
    f"{MAAS_API_URL}/maas-api/v1/models",
    headers={"Authorization": f"Bearer {TOKEN}"}
)

print("Status:", response.status_code)
if response.ok:
    print_json(response.text)
else:
    print(response.text)

## Step 5 — Chat with a model

The MaaS API is OpenAI-compatible, so we can use the `openai` Python library by pointing it at the MaaS endpoint.

In [ ]:
!uv pip install openai -q

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://maas.apps.ocp.cloud.rhai-tmm.dev/prelude-maas/qwen36-27b/v1",
    api_key=TOKEN
)

response = client.chat.completions.create(
    model="qwen36-27b",
    messages=[{"role": "user", "content": "Hello! What can you help me with?"}]
)

print(response.choices[0].message.content)

## Step 6 — Connect to S3 storage (MinIO)

MinIO is an S3-compatible object store. The default credentials for this environment are **minio / minio1234**.

Update `MINIO_ENDPOINT` below to match your MinIO service URL.

In [ ]:
!uv pip install boto3 -q

In [ ]:
import boto3

MINIO_ENDPOINT   = "http://minio.minio.svc.cluster.local:9000"
MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio1234"

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-2"
)

buckets = s3.list_buckets().get("Buckets", [])
print("Existing buckets:", [b["Name"] for b in buckets])

if "models" not in [b["Name"] for b in buckets]:
    s3.create_bucket(Bucket="models")
    print("Created bucket: models")
else:
    print("Bucket 'models' already exists")